In [1]:
import numpy as np

# Structure:
# [[TN, FP],
#  [FN, TP]]
confusion_matrices_single = np.array([
    [[133, 5],
    [0, 35]],
    [[130, 8],
     [0, 35]],
    [[133, 5],
     [0, 35]],
    [[132, 6],
     [0, 35]],
    [[133, 5],
     [0, 35]]
])

confusion_matrices_cctx = np.array([
    [[173, 1],
     [1, 12]],
    [[173, 1],
     [1, 12]],
    [[173, 1],
     [1, 12]],
    [[173, 1],
     [1, 12]],
    [[173, 1],
     [2, 11]],
])

In [2]:
def metrics_from_cm(cm):
    """
    Returns dict of arrays (length N) for accuracy, precision/recall/f1 per class.
    """
    tn = cm[:, 0, 0]
    fp = cm[:, 0, 1]
    fn = cm[:, 1, 0]
    tp = cm[:, 1, 1]

    total = tn + fp + fn + tp
    accuracy = (tp + tn) / total

    # Positive class
    precision_pos = tp / (tp + fp)
    recall_pos    = tp / (tp + fn)
    f1_pos        = 2 * precision_pos * recall_pos / (precision_pos + recall_pos)

    # Negative class (treat "negative" as the positive class of a flipped problem)
    precision_neg = tn / (tn + fn)
    recall_neg    = tn / (tn + fp)
    f1_neg        = 2 * precision_neg * recall_neg / (precision_neg + recall_neg)

    return {
        "accuracy": accuracy,
        "precision_neg": precision_neg, "recall_neg": recall_neg, "f1_neg": f1_neg,
        "precision_pos": precision_pos, "recall_pos": recall_pos, "f1_pos": f1_pos,
    }

def report(name, metrics_dict):
    print(f"--- {name} ---")
    for k, v in metrics_dict.items():
        print(f"{k:15s} mean={np.mean(v):.4f}  std={np.std(v):.4f}")
    print()

# Per-scope
m_cctx = metrics_from_cm(confusion_matrices_cctx)
m_single = metrics_from_cm(confusion_matrices_single)
report("CCTX", m_cctx)
report("SINGLE", m_single)

--- CCTX ---
accuracy        mean=0.9882  std=0.0021
precision_neg   mean=0.9931  std=0.0023
recall_neg      mean=0.9943  std=0.0000
f1_neg          mean=0.9937  std=0.0011
precision_pos   mean=0.9218  std=0.0026
recall_pos      mean=0.9077  std=0.0308
f1_pos          mean=0.9145  std=0.0172

--- SINGLE ---
accuracy        mean=0.9665  std=0.0067
precision_neg   mean=1.0000  std=0.0000
recall_neg      mean=0.9580  std=0.0085
f1_neg          mean=0.9785  std=0.0044
precision_pos   mean=0.8585  std=0.0238
recall_pos      mean=1.0000  std=0.0000
f1_pos          mean=0.9237  std=0.0140



In [3]:

# ---- Option A: sample-weighted global (recommended) ----
# Sum matching experiment-index CMs elementwise -> 5 "combined" CMs
combined_cm = confusion_matrices_cctx + confusion_matrices_single
m_combined = metrics_from_cm(combined_cm)
report("GLOBAL (sample-weighted, sum of CMs per experiment)", m_combined)

--- GLOBAL (sample-weighted, sum of CMs per experiment) ---
accuracy        mean=0.9778  std=0.0030
precision_neg   mean=0.9961  std=0.0013
recall_neg      mean=0.9782  std=0.0037
f1_neg          mean=0.9871  std=0.0018
precision_pos   mean=0.8736  std=0.0182
recall_pos      mean=0.9750  std=0.0083
f1_pos          mean=0.9214  std=0.0098



In [4]:

# ---- Option B: simple average of the two scopes' metrics (unweighted) ----
m_unweighted = {
    k: (m_cctx[k] + m_single[k]) / 2
    for k in m_cctx
}
report("GLOBAL (unweighted average of scope metrics)", m_unweighted)

--- GLOBAL (unweighted average of scope metrics) ---
accuracy        mean=0.9774  std=0.0032
precision_neg   mean=0.9966  std=0.0011
recall_neg      mean=0.9761  std=0.0042
f1_neg          mean=0.9861  std=0.0021
precision_pos   mean=0.8902  std=0.0115
recall_pos      mean=0.9538  std=0.0154
f1_pos          mean=0.9191  std=0.0090

